<a href="https://colab.research.google.com/github/tatyanalitvin/python_for_ds_tasks/blob/dev/HW_2_2_%D0%9D%D0%B5%D0%B7%D0%B1%D0%B0%D0%BB%D0%B0%D0%BD%D1%81%D0%BE%D0%B2%D0%B0%D0%BD%D0%B0_%D0%B1%D0%B0%D0%B3%D0%B0%D1%82%D0%BE%D0%BA%D0%BB%D0%B0%D1%81%D0%BE%D0%B2%D0%B0_%D0%BA%D0%BB%D0%B0%D1%81%D0%B8%D1%84%D1%96%D0%BA%D0%B0%D1%86%D1%96%D1%8F.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.impute import SimpleImputer, MissingIndicator
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

from imblearn.over_sampling import SMOTE, SMOTENC
from imblearn.combine import SMOTETomek
from imblearn.pipeline import Pipeline as ImbPipeline

from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import classification_report, precision_score, recall_score
from sklearn.preprocessing import PolynomialFeatures

from IPython.display import display

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


У цьому ДЗ ми потренуємось розв'язувати задачу багатокласової класифікації за допомогою логістичної регресії з використанням стратегій One-vs-Rest та One-vs-One, оцінити якість моделей та порівняти стратегії.

### Опис задачі і даних

**Контекст**

В цьому ДЗ ми працюємо з даними про сегментацію клієнтів.

Сегментація клієнтів – це практика поділу бази клієнтів на групи індивідів, які схожі між собою за певними критеріями, що мають значення для маркетингу, такими як вік, стать, інтереси та звички у витратах.

Компанії, які використовують сегментацію клієнтів, виходять з того, що кожен клієнт є унікальним і що їхні маркетингові зусилля будуть більш ефективними, якщо вони орієнтуватимуться на конкретні, менші групи зі зверненнями, які ці споживачі вважатимуть доречними та які спонукатимуть їх до купівлі. Компанії також сподіваються отримати глибше розуміння уподобань та потреб своїх клієнтів з метою виявлення того, що кожен сегмент цінує найбільше, щоб точніше адаптувати маркетингові матеріали до цього сегменту.

**Зміст**.

Автомобільна компанія планує вийти на нові ринки зі своїми існуючими продуктами (P1, P2, P3, P4 і P5). Після інтенсивного маркетингового дослідження вони дійшли висновку, що поведінка нового ринку схожа на їхній існуючий ринок.

На своєму існуючому ринку команда з продажу класифікувала всіх клієнтів на 4 сегменти (A, B, C, D). Потім вони здійснювали сегментовані звернення та комунікацію з різними сегментами клієнтів. Ця стратегія працювала для них надзвичайно добре. Вони планують використати ту саму стратегію на нових ринках і визначили 2627 нових потенційних клієнтів.

Ви маєте допомогти менеджеру передбачити правильну групу для нових клієнтів.

В цьому ДЗ використовуємо дані `customer_segmentation_train.csv`[скачати дані](https://drive.google.com/file/d/1VU1y2EwaHkVfr5RZ1U4MPWjeflAusK3w/view?usp=sharing). Це `train.csv`з цього [змагання](https://www.kaggle.com/datasets/abisheksudarshan/customer-segmentation/data?select=train.csv)

**Завдання 1.** Завантажте та підготуйте датасет до аналізу. Виконайте обробку пропущених значень та необхідне кодування категоріальних ознак. Розбийте на тренувальну і тестувальну вибірку, де в тесті 20%. Памʼятаємо, що весь препроцесинг ліпше все ж тренувати на тренувальній вибірці і на тестувальній лише використовувати вже натреновані трансформери.
Але в даному випадку оскільки значень в категоріях небагато, можна зробити обробку і на оригінальних даних, а потім розбити - це простіше. Можна також реалізувати процесинг і тренування моделі з пайплайнами. Обирайте як вам зручніше.

In [3]:
path = f"/content/drive/MyDrive/Colab Notebooks/ML for People/Data/customer_segmentation_train.csv"
customer_segmentation_df = pd.read_csv(path).set_index("ID")
customer_segmentation_df

,Gender,Ever_Married,Age,Graduated,Profession,Work_Experience,Spending_Score,Family_Size,Var_1,Segmentation
ID,,,,,,,,,,
462809,Male,No,22,No,Healthcare,1.0,Low,4.0,Cat_4,D
462643,Female,Yes,38,Yes,Engineer,NaN,Average,3.0,Cat_4,A
466315,Female,Yes,67,Yes,Engineer,1.0,Low,1.0,Cat_6,B
461735,Male,Yes,67,Yes,Lawyer,0.0,High,2.0,Cat_6,B
462669,Female,Yes,40,Yes,Entertainment,NaN,High,6.0,Cat_6,A
...,...,...,...,...,...,...,...,...,...,...
464018,Male,No,22,No,NaN,0.0,Low,7.0,Cat_1,D
464685,Male,No,35,No,Executive,3.0,Low,4.0,Cat_4,D
465406,Female,No,33,Yes,Healthcare,1.0,Low,1.0,Cat_6,D


In [4]:
customer_segmentation_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 8068 entries, 462809 to 461879
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Gender           8068 non-null   object 
 1   Ever_Married     7928 non-null   object 
 2   Age              8068 non-null   int64  
 3   Graduated        7990 non-null   object 
 4   Profession       7944 non-null   object 
 5   Work_Experience  7239 non-null   float64
 6   Spending_Score   8068 non-null   object 
 7   Family_Size      7733 non-null   float64
 8   Var_1            7992 non-null   object 
 9   Segmentation     8068 non-null   object 
dtypes: float64(2), int64(1), object(7)
memory usage: 693.3+ KB


In [5]:
customer_segmentation_df.isnull().sum().sort_values(ascending=False)

,0
Work_Experience,829
Family_Size,335
Ever_Married,140
Profession,124
Graduated,78
Var_1,76
Gender,0
Age,0
Spending_Score,0
Segmentation,0


In [6]:
customer_segmentation_df.describe().round(2)

,Age,Work_Experience,Family_Size
count,8068.00,7239.00,7733.00
mean,43.47,2.64,2.85
std,16.71,3.41,1.53
min,18.00,0.00,1.00
25%,30.00,0.00,2.00
50%,40.00,1.00,3.00
75%,53.00,4.00,4.00
max,89.00,14.00,9.00


In [7]:
# DEFINING COLUMNS
numeric_cols = customer_segmentation_df.select_dtypes(include="number").columns.tolist()
categorical_cols = customer_segmentation_df.select_dtypes("object").columns[:-1].tolist()
target_col = customer_segmentation_df.columns[-1]

numeric_cols_with_nulls = [col for col in numeric_cols if customer_segmentation_df[col].isnull().any()]
categorical_cols_with_nulls = [col for col in categorical_cols if customer_segmentation_df[col].isnull().any()]

ordinal_cols = {
    "Spending_Score": ["Low", "Average", "High"]
}
nominal_cols = [col for col in categorical_cols if col not in ordinal_cols.keys()]

print(f"""
target_col: {target_col}

numeric_cols:
{numeric_cols}

categorical_cols:
{categorical_cols}

    ordinal_cols:
    {ordinal_cols}

    nominal_cols:
    {nominal_cols}
""")




target_col: Segmentation

numeric_cols:
['Age', 'Work_Experience', 'Family_Size']

categorical_cols:
['Gender', 'Ever_Married', 'Graduated', 'Profession', 'Spending_Score', 'Var_1']

    ordinal_cols:
    {'Spending_Score': ['Low', 'Average', 'High']}

    nominal_cols:
    ['Gender', 'Ever_Married', 'Graduated', 'Profession', 'Var_1']



In [8]:
# SPLITTING DATA
train_df, test_df = train_test_split(
    customer_segmentation_df,
    test_size=0.20,
    random_state=42,
    stratify=customer_segmentation_df[target_col]
    )

X_train = train_df.drop(columns=[target_col])
y_train = train_df[target_col]

X_test = test_df.drop(columns=[target_col])
y_test = test_df[target_col]

print("TRAI")
display(X_train.head())
print("TEST")
display(X_test.head())

TRAI


,Gender,Ever_Married,Age,Graduated,Profession,Work_Experience,Spending_Score,Family_Size,Var_1
ID,,,,,,,,,
465905,Female,No,32,Yes,Artist,9.0,Low,1.0,Cat_6
462903,Male,Yes,72,Yes,Entertainment,NaN,Average,2.0,Cat_6
467901,Female,No,33,Yes,Entertainment,1.0,Low,4.0,Cat_6
463613,Female,Yes,48,Yes,Artist,0.0,Average,6.0,Cat_6
459859,Female,Yes,28,No,Doctor,9.0,Low,1.0,Cat_7


TEST


,Gender,Ever_Married,Age,Graduated,Profession,Work_Experience,Spending_Score,Family_Size,Var_1
ID,,,,,,,,,
462588,Male,Yes,41,No,Entertainment,12.0,Average,3.0,Cat_4
459767,Male,No,21,No,Healthcare,4.0,Low,4.0,Cat_6
463475,Female,Yes,36,Yes,Doctor,0.0,Low,1.0,Cat_6
465993,Female,No,27,No,Engineer,0.0,Low,2.0,Cat_6
459016,Female,No,18,No,Healthcare,0.0,Low,6.0,Cat_6


In [9]:
# PREPROCESSING
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(
        strategy = "median",
        add_indicator = True
        )),
    ("scaler", RobustScaler())
])

ordinal_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(
        strategy = "constant",
        fill_value = "missing"
        )),
    ("ordinal", OrdinalEncoder(
        categories = [ordinal_cols["Spending_Score"]],
        handle_unknown = "use_encoded_value",
        unknown_value = -1,
        dtype = np.float64
        )),
    ("scaler", RobustScaler())  # на виході після попереднього кроку буде 0, 1, 2, тож не впевнена зайве зробила чи ні
])

nominal_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(
        strategy = "most_frequent",
        add_indicator = True
        )),
    ("onehot", OneHotEncoder(
        handle_unknown = "ignore",
        sparse_output = False,
        drop = "if_binary",
        dtype = np.float64
    ))
])

categorical_transformer = ColumnTransformer(
    transformers=[
        ("ordinal", ordinal_transformer, list(ordinal_cols.keys())),
        ("nominal", nominal_transformer, nominal_cols)
    ],
    remainder="drop",
    verbose_feature_names_out=False
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numerical", numeric_transformer, numeric_cols),
        ("categorical", categorical_transformer, categorical_cols),
    ],
    remainder="drop",
    verbose_feature_names_out=False
)
preprocessor.set_output(transform="pandas")

ColumnTransformer(transformers=[('numerical',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(add_indicator=True,
                                                                strategy='median')),
                                                 ('scaler', RobustScaler())]),
                                 ['Age', 'Work_Experience', 'Family_Size']),
                                ('categorical',
                                 ColumnTransformer(transformers=[('ordinal',
                                                                  Pipeline(steps=[('imputer',
                                                                                   SimpleImputer(fill_value='missing',
                                                                                                 strategy='constant')),
                                                                                  ('ordinal',
                                                                                   Ord...
                                                                                   SimpleImputer(add_indicator=True,
                                                                                                 strategy='most_frequent')),
                                                                                  ('onehot',
                                                                                   OneHotEncoder(drop='if_binary',
                                                                                                 handle_unknown='ignore',
                                                                                                 sparse_output=False))]),
                                                                  ['Gender',
                                                                   'Ever_Married',
                                                                   'Graduated',
                                                                   'Profession',
                                                                   'Var_1'])],
                                                   verbose_feature_names_out=False),
                                 ['Gender', 'Ever_Married', 'Graduated',
                                  'Profession', 'Spending_Score', 'Var_1'])],
                  verbose_feature_names_out=False)

In [10]:
X_train_preprocessed = preprocessor.fit_transform(X_train)
print("TRAIN")
display(X_train_preprocessed.head())

X_test_preprocessed = preprocessor.transform(X_test)
print("TEST")
display(X_test_preprocessed.head())


TRAIN


,Age,Work_Experience,Family_Size,missingindicator_Work_Experience,missingindicator_Family_Size,Spending_Score,Gender_Male,Ever_Married_Yes,Graduated_Yes,Profession_Artist,...,Var_1_Cat_2,Var_1_Cat_3,Var_1_Cat_4,Var_1_Cat_5,Var_1_Cat_6,Var_1_Cat_7,missingindicator_Ever_Married_True,missingindicator_Graduated_True,missingindicator_Profession_True,missingindicator_Var_1_True
ID,,,,,,,,,,,,,,,,,,,,,
465905,-0.409091,2.00,-1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
462903,1.409091,0.00,-0.5,1.0,0.0,1.0,1.0,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
467901,-0.363636,0.00,0.5,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
463613,0.318182,-0.25,1.5,0.0,0.0,1.0,0.0,1.0,1.0,1.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
459859,-0.590909,2.00,-1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0


TEST


,Age,Work_Experience,Family_Size,missingindicator_Work_Experience,missingindicator_Family_Size,Spending_Score,Gender_Male,Ever_Married_Yes,Graduated_Yes,Profession_Artist,...,Var_1_Cat_2,Var_1_Cat_3,Var_1_Cat_4,Var_1_Cat_5,Var_1_Cat_6,Var_1_Cat_7,missingindicator_Ever_Married_True,missingindicator_Graduated_True,missingindicator_Profession_True,missingindicator_Var_1_True
ID,,,,,,,,,,,,,,,,,,,,,
462588,0.000000,2.75,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
459767,-0.909091,0.75,0.5,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
463475,-0.227273,-0.25,-1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
465993,-0.636364,-0.25,-0.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
459016,-1.045455,-0.25,1.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0


**Завдання 2. Важливо уважно прочитати все формулювання цього завдання до кінця!**

Застосуйте методи ресемплингу даних SMOTE та SMOTE-Tomek з бібліотеки imbalanced-learn до тренувальної вибірки. В результаті у Вас має вийти 2 тренувальних набори: з апсемплингом зі SMOTE, та з ресамплингом з SMOTE-Tomek.

Увага! В нашому наборі даних є як категоріальні дані, так і звичайні числові. Базовий SMOTE не буде правильно працювати з категоріальними даними, але є його модифікація, яка буде. Тому в цього завдання є 2 виконання

  1. Застосувати SMOTE базовий лише на НЕкатегоріальних ознаках.

  2. Переглянути інформацію про метод [SMOTENC](https://imbalanced-learn.org/dev/references/generated/imblearn.over_sampling.SMOTENC.html#imblearn.over_sampling.SMOTENC) і використати цей метод в цій задачі. За цей спосіб буде +3 бали за це завдання і він рекомендований для виконання.

  **Підказка**: аби скористатись SMOTENC треба створити змінну, яка містить індекси ознак, які є категоріальними (їх номер серед колонок) і передати при ініціації екземпляра класу `SMOTENC(..., categorical_features=cat_feature_indeces)`.
  
  Ви також можете розглянути варіант використання варіації SMOTE, який працює ЛИШЕ з категоріальними ознаками [SMOTEN](https://imbalanced-learn.org/dev/references/generated/imblearn.over_sampling.SMOTEN.html)

In [11]:
y_train.value_counts().to_frame("count").style.background_gradient(cmap="RdYlGn")

,count
Segmentation,
D,1814
A,1578
C,1576
B,1486


In [12]:
X_train_preprocessed.columns.tolist()

['Age',
 'Work_Experience',
 'Family_Size',
 'missingindicator_Work_Experience',
 'missingindicator_Family_Size',
 'Spending_Score',
 'Gender_Male',
 'Ever_Married_Yes',
 'Graduated_Yes',
 'Profession_Artist',
 'Profession_Doctor',
 'Profession_Engineer',
 'Profession_Entertainment',
 'Profession_Executive',
 'Profession_Healthcare',
 'Profession_Homemaker',
 'Profession_Lawyer',
 'Profession_Marketing',
 'Var_1_Cat_1',
 'Var_1_Cat_2',
 'Var_1_Cat_3',
 'Var_1_Cat_4',
 'Var_1_Cat_5',
 'Var_1_Cat_6',
 'Var_1_Cat_7',
 'missingindicator_Ever_Married_True',
 'missingindicator_Graduated_True',
 'missingindicator_Profession_True',
 'missingindicator_Var_1_True']

In [13]:
numeric_cols_preprocessed = X_train_preprocessed.select_dtypes(include="number").columns.tolist()
categorical_cols_preprocessed = X_train_preprocessed.select_dtypes("object").columns.tolist()
print(f"""
numeric_cols_preprocessed:
{numeric_cols_preprocessed}

categorical_cols_preprocessed:
{categorical_cols_preprocessed}
""")



numeric_cols_preprocessed:
['Age', 'Work_Experience', 'Family_Size', 'missingindicator_Work_Experience', 'missingindicator_Family_Size', 'Spending_Score', 'Gender_Male', 'Ever_Married_Yes', 'Graduated_Yes', 'Profession_Artist', 'Profession_Doctor', 'Profession_Engineer', 'Profession_Entertainment', 'Profession_Executive', 'Profession_Healthcare', 'Profession_Homemaker', 'Profession_Lawyer', 'Profession_Marketing', 'Var_1_Cat_1', 'Var_1_Cat_2', 'Var_1_Cat_3', 'Var_1_Cat_4', 'Var_1_Cat_5', 'Var_1_Cat_6', 'Var_1_Cat_7', 'missingindicator_Ever_Married_True', 'missingindicator_Graduated_True', 'missingindicator_Profession_True', 'missingindicator_Var_1_True']

categorical_cols_preprocessed:
[]



In [14]:
# Застосовуємо SMOTE
# Я запроцессила всі категоріальні encoder'om and scaler'om, в наступній тасці цікаво, як воно заперформить :D
smote = SMOTE(
    random_state = 42,
    k_neighbors = 5, # не впевнена як визначати скільки сусідів брати для такого датасету, взяла 5 але не знаю чи то багато чи ок
    sampling_strategy = "auto",
    )

X_train_smote, y_train_smote = smote.fit_resample(X_train_preprocessed, y_train)

# imbalance не має конвертувальника в df із коробки, треба вручну
X_train_smote = pd.DataFrame(X_train_smote, columns = X_train_preprocessed.columns)
y_train_smote = pd.Series(y_train_smote, name = y_train.name)

y_train_smote.value_counts()

,count
Segmentation,
A,1814
B,1814
C,1814
D,1814


In [15]:
# Також хочу застосувати SMOTENC, для цього трохи інший препроцесінг, той вибивав помилку тому прибрала тут scaling
numeric_not_scaled = Pipeline(steps=[
    ("imputer", SimpleImputer(
        strategy = "median",
        add_indicator = True,
        )),
    ])
ordinal_not_scaled = Pipeline(steps=[
    ("imputer", SimpleImputer(
        strategy = "constant",
        fill_value = "missing",
        add_indicator = True,
    )),
    ("ordinal", OrdinalEncoder(
        categories = [ordinal_cols["Spending_Score"]],
        handle_unknown = "use_encoded_value",
        unknown_value = -1,
        dtype = np.float64,
    )),
])

# використовую тут ордінал, на відміну від першого процессора, що написала з one hot для номінальних колонок
nominal_transformer = Pipeline(steps = [
    ("imputer", SimpleImputer(
        strategy = "most_frequent",
        add_indicator=True
    )),
    ("ordinal", OrdinalEncoder(
        handle_unknown = "use_encoded_value",
        unknown_value = -1,
    )),
])

preprocessor_smotenc = ColumnTransformer(
    transformers = [
        ("numerical", numeric_not_scaled, numeric_cols),
        ("ordinal", ordinal_not_scaled, list(ordinal_cols.keys())),
        ("nominal", nominal_transformer, nominal_cols),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

# індекси після preprocessor_smotenc output:
feature_names = preprocessor_smotenc.fit(X_train).get_feature_names_out()
categorical_feature_indexes = [i for i, name in enumerate(feature_names)
           if not any(name.startswith(nc) for nc in numeric_cols)]

smotenc = SMOTENC(
    categorical_features=categorical_feature_indexes,
    random_state = 42,
    k_neighbors = 5,
    sampling_strategy = "auto",  # Балансує ВСІ класи до розміру найбільшого
)

pipeline_smotenc = ImbPipeline(steps=[
    ("pre_smote", preprocessor_smotenc),
    ("smotenc", smotenc),
])

X_train_smotenc, y_train_smotenc = pipeline_smotenc.fit_resample(X_train, y_train)
X_test_smotenc = pipeline_smotenc.named_steps["pre_smote"].transform(X_test)
y_test_smotenc = y_test

# imbalance не має конвертувальника в df із коробки, треба вручну
feature_names = pipeline_smotenc.named_steps["pre_smote"].get_feature_names_out()
X_train_smotenc = pd.DataFrame(X_train_smotenc, columns=feature_names)
y_train_smotenc = pd.Series(y_train_smotenc, name=y_train.name)

X_test_smotenc = pd.DataFrame(X_test_smotenc, columns=feature_names)
y_test_smotenc = pd.Series(y_test_smotenc, name=y_train.name)

# Додаю scaler поза пайплайном, бо в пайплайні не працює, а без нього виглядає наче SMOTENC перформить гірше від інших підходів у наступній тасці
numeric_out = X_train_smotenc.select_dtypes(include="number").columns
scaler = StandardScaler()
X_train_smotenc[numeric_out] = scaler.fit_transform(X_train_smotenc[numeric_out])
X_test_smotenc[numeric_out] = scaler.transform(X_test_smotenc[numeric_out])

y_train_smotenc.value_counts()

,count
Segmentation,
A,1814
B,1814
C,1814
D,1814


In [16]:
# Застосовуємо SMOTETomek
smotetomek = SMOTETomek(
    smote = smote,
    random_state = 42,
    n_jobs = -1 # використовує ВСІ доступні ядра процессора
)

X_train_smotetomek, y_train_smotetomek = smotetomek.fit_resample(X_train_preprocessed, y_train)

# imbalance не має конвертувальника в df із коробки, треба вручну
X_train_smotetomek = pd.DataFrame(X_train_smotetomek, columns=X_train_preprocessed.columns)
y_train_smotetomek = pd.Series(y_train_smotetomek, name=y_train.name)

y_train_smotetomek.value_counts()

,count
Segmentation,
D,1500
C,1476
A,1442
B,1430


**Завдання 3**.
  1. Навчіть модель логістичної регресії з використанням стратегії One-vs-Rest з логістичною регресією на оригінальних даних, збалансованих з SMOTE, збалансованих з Smote-Tomek.  
  2. Виміряйте якість кожної з натренованих моделей використовуючи `sklearn.metrics.classification_report`.
  3. Напишіть, яку метрику ви обрали для порівняння моделей.
  4. Яка модель найкраща?
  5. Якщо немає суттєвої різниці між моделями - напишіть свою гіпотезу, чому?

In [17]:
def train_eval_ovr(name, X_train, y_train, X_test, y_test):
    ovr_clf = OneVsRestClassifier(
        LogisticRegression(
            random_state = 42,
            max_iter = 1000,
            solver = "lbfgs"
        )
    )
    ovr_clf.fit(X_train, y_train)

    y_pred_train = ovr_clf.predict(X_train)
    y_pred_test = ovr_clf.predict(X_test)

    y_proba = ovr_clf.predict_proba(X_test)

    print(f"""\n===== {name} =====
    TRAIN
    {classification_report(y_train, y_pred_train)}
    TEST
    {classification_report(y_test, y_pred_test)}
    \n""")

    return ovr_clf, y_pred_test, y_proba


In [18]:
#Original
model_original, pred_original, proba_original = train_eval_ovr(
    "Original",
    X_train_preprocessed, y_train,
    X_test_preprocessed, y_test
)

# SMOTE
model_smote, pred_smote, proba_smote = train_eval_ovr(
    "SMOTE",
    X_train_smote, y_train_smote,
    X_test_preprocessed, y_test
)

# SMOTENC
model_smotenc, pred_smotenc, proba_smotenc = train_eval_ovr(
    "SMOTENC",
    X_train_smotenc, y_train_smotenc,
    X_test_smotenc, y_test_smotenc
)

# SMOTE Tomek
model_smotetomek, pred_smotetomek, proba_smotetomek = train_eval_ovr(
    "SMOTE-Tomek",
    X_train_smotetomek, y_train_smotetomek,
    X_test_preprocessed, y_test
)


===== Original =====
    TRAIN
                  precision    recall  f1-score   support

           A       0.43      0.47      0.45      1578
           B       0.41      0.15      0.22      1486
           C       0.48      0.65      0.55      1576
           D       0.63      0.72      0.67      1814

    accuracy                           0.51      6454
   macro avg       0.49      0.50      0.47      6454
weighted avg       0.49      0.51      0.48      6454

    TEST
                  precision    recall  f1-score   support

           A       0.42      0.45      0.43       394
           B       0.37      0.13      0.20       372
           C       0.49      0.63      0.55       394
           D       0.64      0.76      0.70       454

    accuracy                           0.51      1614
   macro avg       0.48      0.50      0.47      1614
weighted avg       0.49      0.51      0.48      1614

    


===== SMOTE =====
    TRAIN
                  precision    recall  f1-scor

Для порівняння моделей звертала увагу на здебільшого на f1, бо вона враховує і precision і recall

Перформанс виглядає так собі для усіх моделей, особливо для категорії B (може треба якось модель банити за неправильні відповіді у цій категорії, або ж вона мало чим відрізняється від інших і тому модель помиляється, треба подивитись).
Різниця між моделями невелика, оскільки:
- класи у даних майже збалансовані, тому ресемплінг трохи покращує роботу з класом B, але не те щоб дуже
- логістична регресія тут є лінійною, треба спробувати використати складнішу модельку

In [19]:
# Переписала, щоб спробувати те саме з поліномом та трошки кольоровішим output
def train_eval_ovr(name, X_train, y_train, X_test, y_test, poly_degree=2):
    if poly_degree:
        poly = PolynomialFeatures(degree=poly_degree, interaction_only=True, include_bias=False)
        X_train = poly.fit_transform(X_train)
        X_test = poly.transform(X_test)

    ovr_clf = OneVsRestClassifier(
        LogisticRegression(
            random_state=42,
            max_iter=1000,
            solver="lbfgs"
        )
    )
    ovr_clf.fit(X_train, y_train)

    y_pred_train = ovr_clf.predict(X_train)
    y_pred_test = ovr_clf.predict(X_test)
    y_proba = ovr_clf.predict_proba(X_test)

    # Convert reports to DataFrames
    train_report = pd.DataFrame(
        classification_report(y_train, y_pred_train, output_dict=True)
    ).T.round(3)
    test_report = pd.DataFrame(
        classification_report(y_test, y_pred_test, output_dict=True)
    ).T.round(3)

    cmap_1 = "RdYlGn"
    cmap_2 = "viridis"

    print(f"\n===== {name} =====")
    print("TRAIN")
    display(train_report.style
        .background_gradient(
            cmap=cmap_1,
            subset=["precision", "recall", "f1-score"],
            vmin=0, vmax=1)
        .format(precision=3)
    )
    print("TEST")
    display(test_report.style
        .background_gradient(
            cmap=cmap_2,
            subset=["precision", "recall", "f1-score"],
            vmin=0, vmax=1)
        .format(precision=3)
    )

    return ovr_clf, y_pred_test, y_proba

In [20]:
#Original
model_original, pred_original, proba_original = train_eval_ovr(
    "Original",
    X_train_preprocessed, y_train,
    X_test_preprocessed, y_test
)

# SMOTE
model_smote, pred_smote, proba_smote = train_eval_ovr(
    "SMOTE",
    X_train_smote, y_train_smote,
    X_test_preprocessed, y_test
)

# SMOTENC
model_smotenc, pred_smotenc, proba_smotenc = train_eval_ovr(
    "SMOTENC",
    X_train_smotenc, y_train_smotenc,
    X_test_smotenc, y_test_smotenc
)

# SMOTE Tomek
model_smotetomek, pred_smotetomek, proba_smotetomek = train_eval_ovr(
    "SMOTE-Tomek",
    X_train_smotetomek, y_train_smotetomek,
    X_test_preprocessed, y_test
)


===== Original =====
TRAIN


,precision,recall,f1-score,support
A,0.496,0.499,0.497,1578.000
B,0.457,0.338,0.388,1486.000
C,0.574,0.636,0.603,1576.000
D,0.666,0.741,0.701,1814.000
accuracy,0.564,0.564,0.564,0.564
macro avg,0.548,0.554,0.548,6454.000
weighted avg,0.554,0.564,0.556,6454.000


TEST


,precision,recall,f1-score,support
A,0.413,0.411,0.412,394.000
B,0.429,0.317,0.365,372.000
C,0.530,0.581,0.554,394.000
D,0.652,0.740,0.693,454.000
accuracy,0.524,0.524,0.524,0.524
macro avg,0.506,0.512,0.506,1614.000
weighted avg,0.513,0.524,0.515,1614.000



===== SMOTE =====
TRAIN


...

,precision,recall,f1-score,support
A,0.383,0.434,0.407,394.000
B,0.355,0.223,0.274,372.000
C,0.534,0.563,0.548,394.000
D,0.654,0.744,0.696,454.000
accuracy,0.504,0.504,0.504,0.504
macro avg,0.481,0.491,0.481,1614.000
weighted avg,0.489,0.504,0.492,1614.000



===== SMOTE-Tomek =====
TRAIN


,precision,recall,f1-score,support
A,0.565,0.592,0.578,1442.000
B,0.518,0.403,0.454,1430.000
C,0.616,0.663,0.639,1476.000
D,0.703,0.767,0.734,1500.000
accuracy,0.609,0.609,0.609,0.609
macro avg,0.601,0.606,0.601,5848.000
weighted avg,0.602,0.609,0.603,5848.000


TEST


,precision,recall,f1-score,support
A,0.403,0.434,0.418,394.000
B,0.421,0.328,0.369,372.000
C,0.535,0.556,0.545,394.000
D,0.664,0.718,0.690,454.000
accuracy,0.519,0.519,0.519,0.519
macro avg,0.506,0.509,0.506,1614.000
weighted avg,0.513,0.519,0.514,1614.000
